In [ ]:
# %% 1. Imports
from dotenv import load_dotenv
load_dotenv()

import json
from pprint import pprint

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from openai import OpenAI
from pydantic import BaseModel
from typing import Optional
my_instructions = """
You're a documentation assistant. Answer the QUESTION using only facts from the CONTEXT.
If the answer isn't in the CONTEXT, set `answer` to null and `found_answer` to false.
Do not explain or suggest other resources in that case.
""".strip()

# %% 2. Config and response model


class RAGResponse(BaseModel):
    answer: str
    found_answer: bool
    
answer=RAGResponse.model_json_schema()
print("cell 2")

# %% 3. LLM helper
openai_client = OpenAI()

def llm_structured(user_prompt, output_type=RAGResponse, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions is not None:
        messages.append({"role": "system", "content": instructions})
    messages.append({"role": "user", "content": user_prompt})

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type,
    )
    return response.output_parsed

# %% 4. Load documents (slow: run once)
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()
print(f"Loaded {len(files)} documents")

documents = [f.parse() for f in files]

# %% 5. Chunk and index (run once)
document_chunks = chunk_documents(documents, size=3000, step=1500)

chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"],
)
chunk_index.fit(document_chunks)

# %% 6. RAG functions
def search(query):
    return chunk_index.search(query, num_results=5)

def build_prompt(query, search_results):
    search_results_json = json.dumps(search_results, indent=2)
    return f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_results_json}
</CONTEXT>
""".strip()

def rag_structured(query,output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(prompt,output_type, instructions=my_instructions)

# %% 7. Test
answer = rag_structured("how do i implement llm as judge?",)
pprint(answer)

cell 2
Loaded 95 documents
RAGResponse(answer='To implement an LLM as a judge, follow these steps:\n\n1. **Set Up Environment**: Install the necessary packages.\n   ```python\n   pip install evidently\n   ```\n\n2. **Import Required Modules**: \n   ```python\n   import pandas as pd\n   import numpy as np\n   from evidently import Dataset\n   from evidently import Report\n   from evidently.llm.templates import BinaryClassificationPromptTemplate\n   ```\n\n3. **Pass Your OpenAI Key**: Set your OpenAI API key as an environment variable to enable LLM usage.\n   ```python\n   import os\n   os.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n   ```\n\n4. **Create Evaluation Dataset**: Prepare a toy Q&A dataset that includes questions, approved target responses, new responses, and manual labels. For example:\n   ```python\n   data = [\n       ["Question?", "Correct response", "New response", "correct/incorrect", "Notes"],\n       # Add more data here\n   ]\n   ```\n\n5. **Add Descriptors to the Datase

In [30]:
class RAGResponse(BaseModel):
    answer: Optional[str]=None
    found_answer: bool
    
answer=RAGResponse.model_json_schema()
pprint(answer)


my_instructions = """
You're a documentation assistant. Answer the QUESTION using only facts from the CONTEXT.
If the answer isn't in the CONTEXT, set `answer` to null and `found_answer` to false.
Do not explain or suggest other resources in that case.
""".strip()

print(my_instructions)
answer=rag_structured('how do i install kafka on windoeze?')
print(answer.answer)
print(answer.found_answer)

print(RAGResponse.model_json_schema())
print(rag_structured.__defaults__)   # should show the class you just defined



{'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
                           'default': None,
                           'title': 'Answer'},
                'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}
You're a documentation assistant. Answer the QUESTION using only facts from the CONTEXT.
If the answer isn't in the CONTEXT, set `answer` to null and `found_answer` to false.
Do not explain or suggest other resources in that case.
The provided context does not include instructions for installing Kafka on Windows.
False
{'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Answer'}, 'found_answer': {'title': 'Found Answer', 'type': 'boolean'}}, 'required': ['found_answer'], 'title': 'RAGResponse', 'type': 'object'}
(<class '__main__.RAGResponse'>,)
